# Task 1 — Wound Region Segmentation: Colab Ablation Run


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the dataset

Unzips the dataset (skipped if already extracted), then finds the
extracted `wound-region-segmentation` folder wherever it landed inside
the zip (its internal layout has drifted before — this searches for it
instead of hardcoding a path) and symlinks it into the repo.

In [ ]:
import glob
import os

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-region-segmentation.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-region-segmentation"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
source = candidates[0]
print("dataset found at:", source)

os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if os.path.islink(DATASET_DIR):
    os.remove(DATASET_DIR)
os.symlink(source, DATASET_DIR)
!ls -la {DATASET_DIR}

## 4. Ensure the `skin` color mapping matches training code

Forces `skin` to `#0000FF` in the dataset's `color-mappings.json`
(matching `datasets/README.md`) — a no-op if it's already correct.

In [ ]:
import json

color_map_path = f"{DATASET_DIR}/color-mappings.json"
with open(color_map_path) as f:
    mapping = json.load(f)
print("before:", mapping)

if mapping.get("skin") != "#0000FF":
    mapping["skin"] = "#0000FF"
    with open(color_map_path, "w") as f:
        json.dump(mapping, f, indent=4)
    print("fixed ->", mapping)
else:
    print("already correct, no change")

## 5. Install dependencies

In [ ]:
import os
os.chdir(f"{REPO_DIR}/task_1")
!pip install -q -r requirements.txt

## 6. Redirect checkpoints/results to Drive

So a killed/disconnected runtime doesn't lose completed-run history or
checkpoints — everything survives on Drive across sessions.

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/task_1_outputs"

with open("configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/ablation_results.csv"
with open("configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 7. Run the ablation grid

Restricted to the paper's top-10 Table 4 configs (advisor call:
the full 69-config x 3-fold grid is unnecessary, these 10 are enough)
via `--configs`. Drop `--configs ...` and it falls back to the full
grid if ever needed again.

`run_ablation.py` already skips completed `(config, fold)` rows,
resumes a killed fold from its last saved epoch, and isolates each
config's failures so one bad config can't stall the rest — but a
crash still kills the Python process itself. This loop restarts it
automatically so a crash doesn't need you to re-run the cell by hand.
Output is streamed line-by-line (`Popen` + `-u`) instead of
`subprocess.run()`, which buffers the whole thing and prints nothing
in Colab until the process exits. A full runtime disconnect kills the
loop too (Colab tears down the kernel); at that point just re-run this
cell — the run picks up from the last saved epoch either way.

In [ ]:
!python check_compatibility.py

In [ ]:
import subprocess
import time

TOP_10_CONFIGS = ",".join([
    "unet__mit_b2__imagenet",
    "unet__resnet50__ssl",
    "unetplusplus__densenet121__imagenet",
    "unet__mit_b1__imagenet",
    "manet__mit_b2__imagenet",
    "fpn__mit_b2__imagenet",
    "unet__densenet121__imagenet",
    "deeplabv3plus__resnet50__swsl",
    "deeplabv3plus__mit_b3__imagenet",
    "unetplusplus__efficientnet-b3__imagenet",
])

cmd = ["python", "-u", "run_ablation.py", "--configs", TOP_10_CONFIGS, "--batch", "64", "--workers", "16"]
while True:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    returncode = proc.wait()
    if returncode == 0:
        print("grid finished")
        break
    print(f"run_ablation.py exited with code {returncode}, retrying in 10s...")
    time.sleep(10)

## 7b. Run the SegFormer-only study (3-fold, ImageNet)

Forgotten in the top-10 run above: SegFormer isn't one of the 5
generic SMP decoders — it uses its own hand-written all-MLP decode
head and is only compatible with the MiT encoder family, ImageNet
weights only (`configs/grid.yaml`'s `segformer` block — SSL/SWSL
aren't available for MiT). This runs all 3 MiT-{B1,B2,B3} SegFormer
configs under the same 3-fold protocol (`base.yaml`'s default
`num_folds: 3`), appending to the same `results/ablation_results.csv`.

In [ ]:
import subprocess
import time

cmd = ["python", "-u", "run_ablation.py", "--study", "segformer_only", "--batch", "64", "--workers", "16"]
while True:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    returncode = proc.wait()
    if returncode == 0:
        print("segformer study finished")
        break
    print(f"run_ablation.py exited with code {returncode}, retrying in 10s...")
    time.sleep(10)

## 8. Preview sample panels

After every completed `(config, fold)`, `run_ablation.py` saves a
3-panel PNG (image / ground truth / prediction overlay) to
`results/samples/`. This shows the most recent ones.

In [ ]:
import glob
from IPython.display import Image, display

samples_dir = f"{OUTPUT_DIR}/results/samples"
paths = sorted(glob.glob(f"{samples_dir}/*.png"))
print(f"{len(paths)} sample panels")
for path in paths[-10:]:
    print(path.split("/")[-1])
    display(Image(filename=path))

## 9. Per-class precision/recall and confusion matrix (saved checkpoints)

Dice/IoU alone doesn't say *where* the errors are: low **wound recall**
means the boundary is under-called (the predicted wound shrinks, so a
downstream crop for Task 2/3 would miss part of the wound); low
**wound precision** means healthy skin is being misclassified as wound.
This runs inference with every checkpoint saved under
`{OUTPUT_DIR}/checkpoints` — whichever configs happen to have saved
checkpoints so far, not just the top-10 — and reports:

- Per-class precision & recall (background/skin/wound), mean±std across
  the saved folds for that config
- The full 3x3 confusion matrix (true class x predicted class), summed
  as raw pixel counts across those same folds

Both tables are written to `{OUTPUT_DIR}/results/precision_recall.csv`
and `.../confusion_matrix.csv`.

In [ ]:
import csv
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.dataset import WoundRegionDataset, load_color_mapping
from src.metrics import compute_confusion, precision_recall_from_confusion
from src.model import build_model, build_segformer
from src.splits import build_folds
from src.utils import get_device, load_yaml

base_cfg = load_yaml("configs/base.yaml")
resolved_by_id = {c["config_id"]: c for c in json.loads(Path("configs/resolved_grid.json").read_text())}
class_names = base_cfg["classes"]  # ["background", "skin", "wound"]
num_classes = len(class_names)

colors = load_color_mapping(DATASET_DIR)
folds = build_folds(DATASET_DIR, base_cfg["num_folds"], base_cfg["val_fraction"], base_cfg["seed"])
device = get_device()

def build_model_from_config(cfg):
    if cfg["architecture"] == "segformer":
        return build_segformer(cfg["encoder"], None, num_classes=num_classes)
    return build_model(cfg["architecture"], cfg["encoder"], None, num_classes=num_classes)

# Group saved checkpoints (not .resume.pt — those are mid-fold, not final) by config_id.
checkpoint_dir = Path(OUTPUT_DIR) / "checkpoints"
ckpt_pattern = re.compile(r"^(?P<config_id>.+)_fold(?P<fold>\d+)\.pt$")
by_config = {}
for p in sorted(checkpoint_dir.glob("*_fold*.pt")):
    m = ckpt_pattern.match(p.name)
    if not m:
        continue
    by_config.setdefault(m.group("config_id"), {})[int(m.group("fold"))] = p
print(f"found checkpoints for {len(by_config)} config(s)")

pr_rows = []
confusion_rows = []

for config_id, fold_ckpts in sorted(by_config.items()):
    if config_id not in resolved_by_id:
        print(f"skip {config_id}: not found in configs/resolved_grid.json")
        continue
    cfg = resolved_by_id[config_id]

    fold_precisions, fold_recalls = [], []
    total_conf = np.zeros((num_classes, num_classes), dtype=np.int64)

    for fold_idx in sorted(fold_ckpts):
        test_ds = WoundRegionDataset(folds[fold_idx]["test"], DATASET_DIR, base_cfg["image_size"], colors)
        test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=4)

        model = build_model_from_config(cfg)
        model.load_state_dict(torch.load(fold_ckpts[fold_idx], map_location=device))
        model.to(device).eval()

        conf = torch.zeros((num_classes, num_classes), dtype=torch.int64, device=device)
        with torch.no_grad():
            for images, targets in test_loader:
                images, targets = images.to(device), targets.to(device)
                conf += compute_confusion(model(images), targets, num_classes)
        conf = conf.cpu().numpy()
        total_conf += conf

        precision, recall = precision_recall_from_confusion(conf)
        fold_precisions.append(precision)
        fold_recalls.append(recall)

        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    fold_precisions = np.array(fold_precisions)  # (n_folds, num_classes)
    fold_recalls = np.array(fold_recalls)

    row = {
        "config_id": config_id, "architecture": cfg["architecture"],
        "encoder": cfg["encoder"], "weights": cfg["weights"], "n_folds": len(fold_ckpts),
    }
    for ci, cname in enumerate(class_names):
        row[f"precision_{cname}_mean"] = round(float(fold_precisions[:, ci].mean()), 4)
        row[f"precision_{cname}_std"] = round(float(fold_precisions[:, ci].std()), 4)
        row[f"recall_{cname}_mean"] = round(float(fold_recalls[:, ci].mean()), 4)
        row[f"recall_{cname}_std"] = round(float(fold_recalls[:, ci].std()), 4)
    pr_rows.append(row)

    conf_row = {"config_id": config_id}
    for i, gt_name in enumerate(class_names):
        for j, pred_name in enumerate(class_names):
            conf_row[f"true_{gt_name}_pred_{pred_name}"] = int(total_conf[i, j])
    confusion_rows.append(conf_row)

    print(f"[{config_id}] done ({len(fold_ckpts)} fold(s))")

out_dir = Path(OUTPUT_DIR) / "results"
out_dir.mkdir(parents=True, exist_ok=True)

pr_path = out_dir / "precision_recall.csv"
with open(pr_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(pr_rows[0].keys()))
    writer.writeheader()
    writer.writerows(pr_rows)
print(f"wrote {pr_path}")

conf_path = out_dir / "confusion_matrix.csv"
with open(conf_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(confusion_rows[0].keys()))
    writer.writeheader()
    writer.writerows(confusion_rows)
print(f"wrote {conf_path}")

# Readable "mean±std" display, paper Table 4/5 style.
display_rows = []
for row in pr_rows:
    d = {"config_id": row["config_id"], "n_folds": row["n_folds"]}
    for cname in class_names:
        d[f"precision_{cname}"] = f"{row[f'precision_{cname}_mean']:.4f}±{row[f'precision_{cname}_std']:.4f}"
        d[f"recall_{cname}"] = f"{row[f'recall_{cname}_mean']:.4f}±{row[f'recall_{cname}_std']:.4f}"
    display_rows.append(d)

display(pd.DataFrame(display_rows).set_index("config_id"))
display(pd.DataFrame(confusion_rows).set_index("config_id"))